# Full Benchmark Evaluation (50 Dev + 100 Test)
### Detecting and Repairing Hallucinations in LLM-Generated Answers

This notebook runs the full evaluation benchmark in an isolated Python 3.12 environment:
- **Dev Set:** 50 records (pipeline + B0)
- **Test Set:** 100 records (pipeline + B0)
- **Index:** FEVER FTS5 Wikipedia lexical index (saved on Google Drive)
- **Model:** Groq API (`qwen/qwen3.8-27b` or `openai/gpt-oss-120b` / `20b`)


In [ ]:
# Step 1: Install uv and Python 3.12 (Colab kernel defaults to 3.13)
import subprocess, sys

print("Installing uv and Python 3.12...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
print("Python 3.12 installed successfully.")


In [ ]:
# Step 2: Clone repository & set up isolated Python 3.12 venv
from pathlib import Path
import sys, os, subprocess, json, shutil, zipfile, io

REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"
REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")

if not REPO.exists():
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    print("Pulling latest commits...")
    subprocess.run(["git", "-C", str(REPO), "pull"], check=True)

os.chdir(REPO)

# Create isolated Python 3.12 virtualenv
ENV = Path("/content/benchmark_env")
PYTHON = str(ENV / "bin/python")

if not Path(PYTHON).exists():
    print("Creating Python 3.12 venv...")
    subprocess.run(["uv", "venv", "--python", "3.12", "--seed", str(ENV)], check=True)
    print("Installing requirements-evaluation.txt...")
    subprocess.run([PYTHON, "-m", "pip", "install", "-r", str(REPO / "requirements-evaluation.txt")], check=True)

# Verify Python version is 3.12
version_check = subprocess.check_output([PYTHON, "-c", "import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip()
assert version_check == "3.12", f"Expected Python 3.12, got {version_check}"
print(f"Isolated Python 3.12 environment ready: {PYTHON}")


In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/hallucination_full_benchmark')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent Google Drive directory:", DRIVE_ROOT)


In [ ]:
# Step 4: Configure Groq API Key & Upload Gold Inputs + Index
from google.colab import files, userdata
import io, zipfile
from pathlib import Path

groq_key = userdata.get("GROQ_API_KEY")
assert groq_key, "GROQ_API_KEY not found in Colab Secrets! Click the key icon on the left sidebar, add GROQ_API_KEY, and toggle Notebook access."
os.environ["GROQ_API_KEY"] = groq_key

GOLD = REPO / "data/gold"
GOLD.mkdir(parents=True, exist_ok=True)
INDEX_DIR = REPO / "indexes"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

dev_gold = GOLD / "dev.jsonl"
test_gold = GOLD / "test.jsonl"
index_file = INDEX_DIR / "fever_fts.sqlite"

if not dev_gold.exists() or not test_gold.exists() or not index_file.exists():
    print("Please upload gold_inputs.zip (from your local results/evaluation/gold_inputs.zip):")
    uploads = files.upload()
    for name, data in uploads.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                for member in zf.namelist():
                    extracted = zf.read(member)
                    if member == "splits.json":
                        (REPO / "data/splits.json").write_bytes(extracted)
                    elif member == "fever_fts.sqlite":
                        index_file.write_bytes(extracted)
                        print("Pre-built FTS5 index extracted directly (8.2 MB)! Skipping download.")
                    else:
                        (GOLD / Path(member).name).write_bytes(extracted)

assert dev_gold.exists(), "data/gold/dev.jsonl missing!"
assert test_gold.exists(), "data/gold/test.jsonl missing!"
print(f"Dev gold records: {len(dev_gold.read_text().splitlines())}")
print(f"Test gold records: {len(test_gold.read_text().splitlines())}")
if index_file.exists():
    print(f"Index ready: {index_file} ({index_file.stat().st_size / (1024**2):.1f} MB)")


In [ ]:
# Step 5: Fast FEVER Download & Direct Zip Stream Index Builder
import subprocess, sys, shutil, os, zipfile
from pathlib import Path

REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")
REPO_URL = "https://github.com/harshitap1305/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers.git"

# Ensure repo exists
if not REPO.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull"], check=False)

PYTHON = sys.executable  # stdlib only — Python 3.13 is fine

INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_ROOT = Path('/content/drive/MyDrive/hallucination_full_benchmark')
DRIVE_INDEX = DRIVE_ROOT / "fever_fts.sqlite"
ALT_DRIVE_INDEX = Path("/content/drive/MyDrive/hallucination_reviewed_pilot/fever_fts.sqlite")
INDEX.parent.mkdir(parents=True, exist_ok=True)

# 1. Reuse index from Drive if already present (saves all download+build time)
if DRIVE_INDEX.exists():
    print(f"Reusing existing index from Drive: {DRIVE_INDEX}")
    if INDEX.is_symlink() or INDEX.exists(): INDEX.unlink()
    shutil.copyfile(DRIVE_INDEX, INDEX)
elif ALT_DRIVE_INDEX.exists():
    print(f"Reusing existing index from pilot Drive: {ALT_DRIVE_INDEX}")
    if INDEX.is_symlink() or INDEX.exists(): INDEX.unlink()
    shutil.copyfile(ALT_DRIVE_INDEX, INDEX)
else:
    archive = Path("/content/fever-wiki-pages.zip")
    # Note: Zenodo wiki-pages.zip is exactly 1.71 GB compressed (expands to ~20 GB)
    MIN_SIZE = 1_700_000_000

    if not archive.exists() or archive.stat().st_size < MIN_SIZE:
        print("Installing aria2...")
        subprocess.run(["apt-get", "install", "-y", "-q", "aria2"], check=True)
        print("Downloading FEVER wiki from Zenodo (1.71 GB, ~2-5 min with 16 connections)...")
        subprocess.run([
            "aria2c",
            "--dir=/content",
            "--out=fever-wiki-pages.zip",
            "-x", "16", "-s", "16", "-k", "1M",
            "https://zenodo.org/records/4925954/files/wiki-pages.zip?download=1"
        ], check=True)

    print(f"Archive verified: {archive.stat().st_size / 1e9:.2f} GB")

    # Build FTS5 index by streaming directly from zip (no unzip = no disk overflow!)
    print("Building SQLite FTS5 index (streaming from zip, ~10-20 min)...")
    subprocess.run([
        PYTHON, "scripts/build_fever_fts.py",
        "--wiki-pages-zip", str(archive),
        "--output", str(INDEX),
        "--overwrite"
    ], check=True, cwd=REPO)

    # Backup to Drive so future runs skip all this
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Backing up index to Google Drive...")
    shutil.copyfile(INDEX, DRIVE_INDEX)

print(f"✓ Index ready: {INDEX} | Size: {INDEX.stat().st_size / (1024**3):.2f} GB")


In [ ]:
# Step 6: Select & Verify Model
import subprocess, sys

# Install groq into the notebook environment so the cell can query models
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "groq"], check=True)

from groq import Groq
client = Groq()
models = [m.id for m in client.models.list().data]
print("Available models on your Groq key:")
for m in sorted(models):
    print(" -", m)

# Default to qwen/qwen3.8-27b or openai/gpt-oss-120b
MODEL = "qwen/qwen3.8-27b" if "qwen/qwen3.8-27b" in models else "openai/gpt-oss-120b"
print(f"
Using model: {MODEL}")


In [ ]:
# Step 7: Run Benchmark on Dev Split (50 records)
import shutil
from pathlib import Path

REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")
ENV  = Path("/content/benchmark_env")
PYTHON312 = str(ENV / "bin/python")

# Auto-rebuild venv if session restarted and ENV is gone
if not Path(PYTHON312).exists():
    print("Rebuilding Python 3.12 venv (session was restarted)...")
    import subprocess, sys
    subprocess.run(["uv", "venv", "--python", "3.12", "--seed", str(ENV)], check=True)
    subprocess.run([PYTHON312, "-m", "pip", "install", "-r",
                    str(REPO / "requirements-evaluation.txt")], check=True)

# Confirm index is accessible
INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_INDEX = Path('/content/drive/MyDrive/hallucination_full_benchmark/fever_fts.sqlite')
if not INDEX.exists() and DRIVE_INDEX.exists():
    INDEX.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DRIVE_INDEX, INDEX)
assert INDEX.exists(), f"FEVER index missing at {INDEX} — run Cell 5 (Step 5) first!"

# Smart resume: delete dir if no manifest (aborted run), pass --resume only if manifest exists
DEV_OUTPUT = REPO / "results/evaluation/full_benchmark_v1/pipeline_dev"
if DEV_OUTPUT.exists() and not (DEV_OUTPUT / "manifest.json").exists():
    print("Removing incomplete run directory (no manifest)...")
    shutil.rmtree(DEV_OUTPUT)
resume_flag = ["--resume"] if (DEV_OUTPUT / "manifest.json").exists() else []

cmd = [
    PYTHON312, "-u", "scripts/run_benchmark.py",
    "--config", "configs/evaluation_halueval_colab_full.yaml",
    "--inputs", "data/gold/dev.jsonl",
    "--splits", "data/splits.json",
    "--output-dir", str(DEV_OUTPUT),
    "--task", "repair",
    "--split", "dev",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", MODEL,
] + resume_flag

print(f"Executing benchmark on Dev split (model: {MODEL}){'  [RESUMING]' if resume_flag else ''}...")
import subprocess
subprocess.run(cmd, check=True, cwd=REPO)
print("Dev execution finished successfully!")


In [ ]:
# Step 8: Run Benchmark on Test Split (100 records)
import shutil, subprocess
from pathlib import Path

REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")
ENV  = Path("/content/benchmark_env")
PYTHON312 = str(ENV / "bin/python")

# Auto-rebuild venv if session restarted and ENV is gone
if not Path(PYTHON312).exists():
    print("Rebuilding Python 3.12 venv (session was restarted)...")
    import sys
    subprocess.run(["uv", "venv", "--python", "3.12", "--seed", str(ENV)], check=True)
    subprocess.run([PYTHON312, "-m", "pip", "install", "-r",
                    str(REPO / "requirements-evaluation.txt")], check=True)

# Confirm index is accessible
INDEX = REPO / "indexes/fever_fts.sqlite"
DRIVE_INDEX = Path('/content/drive/MyDrive/hallucination_full_benchmark/fever_fts.sqlite')
if not INDEX.exists() and DRIVE_INDEX.exists():
    INDEX.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(DRIVE_INDEX, INDEX)
assert INDEX.exists(), f"FEVER index missing at {INDEX} — run Cell 5 (Step 5) first!"

# Smart resume: delete dir if no manifest (aborted run), pass --resume only if manifest exists
TEST_OUTPUT = REPO / "results/evaluation/full_benchmark_v1/pipeline_test"
if TEST_OUTPUT.exists() and not (TEST_OUTPUT / "manifest.json").exists():
    print("Removing incomplete run directory (no manifest)...")
    shutil.rmtree(TEST_OUTPUT)
resume_flag = ["--resume"] if (TEST_OUTPUT / "manifest.json").exists() else []

cmd = [
    PYTHON312, "-u", "scripts/run_benchmark.py",
    "--config", "configs/evaluation_halueval_colab_full.yaml",
    "--inputs", "data/gold/test.jsonl",
    "--splits", "data/splits.json",
    "--output-dir", str(TEST_OUTPUT),
    "--task", "repair",
    "--split", "test",
    "--options-file", "configs/full_benchmark_v1.options.json",
    "--model", MODEL,
] + resume_flag

print(f"Executing benchmark on Test split (model: {MODEL}){'  [RESUMING]' if resume_flag else ''}...")
subprocess.run(cmd, check=True, cwd=REPO)
print("Test execution finished successfully!")


In [ ]:
# Step 9: Build Evaluation Progress Reports & Backup to Drive
import subprocess, zipfile, shutil
from pathlib import Path

REPO = Path("/content/Detecting-and-Repairing-Hallucinations-in-LLM-Generated-Answers")
ENV  = Path("/content/benchmark_env")
PYTHON312 = str(ENV / "bin/python")
DRIVE_ROOT = Path('/content/drive/MyDrive/hallucination_full_benchmark')

DEV_OUTPUT  = REPO / "results/evaluation/full_benchmark_v1/pipeline_dev"
TEST_OUTPUT = REPO / "results/evaluation/full_benchmark_v1/pipeline_test"
DEV_REPORT  = REPO / "results/evaluation/full_benchmark_v1/report_pipeline_dev"
TEST_REPORT = REPO / "results/evaluation/full_benchmark_v1/report_pipeline_test"

# Dev progress report
print("Building Dev progress report...")
subprocess.run([
    PYTHON312, "scripts/build_evaluation_report.py",
    "--run-dir", str(DEV_OUTPUT),
    "--gold", "data/gold/dev.jsonl",
    "--output-dir", str(DEV_REPORT),
    "--mode", "progress"
], check=True, cwd=REPO)

# Test progress report
print("Building Test progress report...")
subprocess.run([
    PYTHON312, "scripts/build_evaluation_report.py",
    "--run-dir", str(TEST_OUTPUT),
    "--gold", "data/gold/test.jsonl",
    "--output-dir", str(TEST_REPORT),
    "--mode", "progress"
], check=True, cwd=REPO)

# Audit both bundles
print("Auditing Dev bundle...")
subprocess.run([PYTHON312, "scripts/audit_baseline.py",
                "--bundle", str(DEV_REPORT), "--gold", "data/gold/dev.jsonl"], check=True, cwd=REPO)
print("Auditing Test bundle...")
subprocess.run([PYTHON312, "scripts/audit_baseline.py",
                "--bundle", str(TEST_REPORT), "--gold", "data/gold/test.jsonl"], check=True, cwd=REPO)

# Package all results (excluding the large sqlite index) into a zip on Drive
ZIP_DEST = DRIVE_ROOT / "full_benchmark_v1_results.zip"
print(f"Creating backup archive at: {ZIP_DEST}")
with zipfile.ZipFile(ZIP_DEST, 'w', zipfile.ZIP_DEFLATED) as zf:
    results_dir = REPO / "results/evaluation/full_benchmark_v1"
    for file_path in results_dir.rglob('*'):
        if file_path.is_file() and not file_path.name.endswith('.sqlite'):
            zf.write(file_path, file_path.relative_to(REPO))

print("Results archive saved to Google Drive:")
print(" ->", ZIP_DEST)
